# Feature engineering — Base 4 — clima de Jena

Orquestra `features/temporal.py` com `BASE_4`.

Passo de `10min`. Horizonte final pendente.

## Disponibilidade

- Alvo provisório: `T (degC)`.
- Demais medições: estado em `t` para o intervalo seguinte.
- `wd (deg)` bruto: **proibido**; usar `wd_sin`/`wd_cos`.
- `-9999` deve ser ausência antes desta etapa.
- Arquivos: `catalogo_features_base4.csv` e `disponibilidade_covariaveis_base4.csv`.


In [1]:
from pathlib import Path
import sys

import pandas as pd

def find_project_root(start):
    for candidate in (start, *start.parents):
        if (candidate / "projeto.yaml").is_file():
            return candidate
    raise FileNotFoundError("Execute a partir do projeto ou de uma subpasta dele.")

ROOT = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(ROOT / "src"))
from series_temporais.features.regras_bases import BASE_4, validar_proibidas
from series_temporais.features.temporal import (
    build_one_step_frame,
    choque_futuro_nao_altera_features,
    chronological_train_test,
)

BASE_DIR = ROOT / "data" / "base_04"
DATA_PATH = BASE_DIR / "prepared.csv"
CATALOGO_PATH = BASE_DIR / "catalogo_features_base4.csv"
DISPONIBILIDADE_PATH = BASE_DIR / "disponibilidade_covariaveis_base4.csv"
SPEC = BASE_4
TRAIN_RATIO = 0.8


## Quadro de um passo

Cada linha é uma origem. O alvo do modelo é a variação até o passo seguinte; ele não entra nas features.

In [2]:
df = pd.read_csv(DATA_PATH)
df = df.sort_values(SPEC.time_col, kind="stable").reset_index(drop=True)
validar_proibidas(SPEC.external_cols(), SPEC)
quadro, features = build_one_step_frame(
    df,
    time_col=SPEC.time_col,
    target_col=SPEC.target_col,
    external_cols=SPEC.external_cols(),
    expected_step=SPEC.expected_step,
    lags=SPEC.lags,
    windows=SPEC.windows,
    availability=SPEC.availability(),
)
assert len(quadro) == 414150
assert len(features) == 33
assert {"y_true", "target_delta", "target_time"}.isdisjoint(features)
quadro[features].head()


,level_t,p (mbar)_t,Tpot (K)_t,Tdew (degC)_t,rh (%)_t,VPmax (mbar)_t,VPact (mbar)_t,VPdef (mbar)_t,sh (g/kg)_t,H2OC (mmol/mol)_t,...,target_mean_144,target_std_144,target_mean_1008,target_std_1008,target_hour_sin,target_hour_cos,target_dow_sin,target_dow_cos,target_month_sin,target_month_cos
0,-10.60,998.67,262.66,-11.64,92.0,2.72,2.50,0.22,1.56,2.50,...,-15.662778,5.877155,-7.613958,6.055611,0.000000,1.000000,0.433884,-0.900969,0.0,1.0
1,-10.72,998.71,262.54,-11.84,91.4,2.69,2.46,0.23,1.53,2.46,...,-15.588403,5.872984,-7.616518,6.056328,0.000000,1.000000,0.433884,-0.900969,0.0,1.0
2,-11.29,998.83,261.96,-12.52,90.5,2.57,2.33,0.24,1.45,2.33,...,-15.517222,5.869442,-7.618810,6.057065,0.000000,1.000000,0.433884,-0.900969,0.0,1.0
3,-11.81,998.96,261.43,-13.09,90.1,2.47,2.22,0.24,1.38,2.22,...,-15.453403,5.865095,-7.621567,6.058102,0.000000,1.000000,0.433884,-0.900969,0.0,1.0
4,-11.78,999.04,261.45,-12.99,90.6,2.47,2.24,0.23,1.40,2.24,...,-15.394722,5.858979,-7.625040,6.059499,0.258819,0.965926,0.433884,-0.900969,0.0,1.0


## Catálogo

O arquivo CSV fica na pasta do grupo e registra, para cada coluna de `X`, se a informação é conhecida na origem, só no passado ou antecipadamente.

In [3]:
catalogo = SPEC.catalogo(features)
disponibilidade = SPEC.tabela_disponibilidade()
assert not catalogo.grupo.eq("revisar").any()
catalogo.to_csv(CATALOGO_PATH, index=False)
disponibilidade.to_csv(DISPONIBILIDADE_PATH, index=False)
disponibilidade


,coluna,disponibilidade,atraso,feature,status,justificativa
0,p (mbar),na_origem,0,p (mbar)_t,em_validacao,Medição simultânea; entra em t para prever o i...
1,Tpot (K),na_origem,0,Tpot (K)_t,em_validacao,Medição simultânea; entra em t para prever o i...
2,Tdew (degC),na_origem,0,Tdew (degC)_t,em_validacao,Medição simultânea; entra em t para prever o i...
3,rh (%),na_origem,0,rh (%)_t,em_validacao,Medição simultânea; entra em t para prever o i...
4,VPmax (mbar),na_origem,0,VPmax (mbar)_t,em_validacao,Medição simultânea; entra em t para prever o i...
5,VPact (mbar),na_origem,0,VPact (mbar)_t,em_validacao,Medição simultânea; entra em t para prever o i...
6,VPdef (mbar),na_origem,0,VPdef (mbar)_t,em_validacao,Medição simultânea; entra em t para prever o i...
7,sh (g/kg),na_origem,0,sh (g/kg)_t,em_validacao,Medição simultânea; entra em t para prever o i...
8,H2OC (mmol/mol),na_origem,0,H2OC (mmol/mol)_t,em_validacao,Medição simultânea; entra em t para prever o i...
9,rho (g/m**3),na_origem,0,rho (g/m**3)_t,em_validacao,Medição simultânea; entra em t para prever o i...


## Corte cronológico

A última linha de treino cujo alvo cruzaria a primeira origem de teste é removida.

In [4]:
treino, teste = chronological_train_test(quadro, TRAIN_RATIO)
assert len(treino) == 331319
assert len(teste) == 82830
assert treino.target_time.max() < teste.origin_time.min()
pd.DataFrame({
    "recorte": ["treino", "teste"],
    "linhas": [len(treino), len(teste)],
    "primeira_origem": [treino.origin_time.min(), teste.origin_time.min()],
    "ultima_origem": [treino.origin_time.max(), teste.origin_time.max()],
})


,recorte,linhas,primeira_origem,ultima_origem
0,treino,331319,2009-01-08 00:10:00,2015-05-26 13:00:00
1,teste,82830,2015-05-26 13:20:00,2016-12-31 23:50:00


## Checagem de leakage

A segunda metade da série recebe um valor artificial. As features das origens anteriores precisam permanecer idênticas.

In [5]:
resultado = choque_futuro_nao_altera_features(
    df,
    time_col=SPEC.time_col,
    indice_inicial_do_choque=len(df) // 2,
    colunas=[SPEC.target_col, *SPEC.external_cols()],
    target_col=SPEC.target_col,
    external_cols=SPEC.external_cols(),
    expected_step=SPEC.expected_step,
    lags=SPEC.lags,
    windows=SPEC.windows,
    availability=SPEC.availability(),
)
assert resultado["iguais"] is True
assert resultado["diferenca_maxima"] == 0
resultado


{'origens_comparadas': 208365, 'diferenca_maxima': 0.0, 'iguais': True}